# 🧬 EdgeMamba-3: Full Training Pipeline

**Edge-Centric State Space Model for Graph Learning**  
*Yajat Kapur · 22BBS0110 · CBS1904 Capstone*

---

This notebook orchestrates the **entire EdgeMamba-3 pipeline** — from environment
validation through full training, evaluation, and ablation studies.

### How It Works
1. You supply the **project root path** (the `edgemamba3/` directory).
2. The notebook dynamically adds it to `sys.path` and imports every module.
3. All four experiment configs (LRGB × 2, RelBench × 2) are loaded from YAML.
4. Training, validation, testing, and report generation run end-to-end.

### Table of Contents
0. [Setup & Imports](#0-setup)
1. [Environment Validation](#1-env)
2. [LRGB Training (Peptides-func & Peptides-struct)](#2-lrgb)
3. [RelBench Training (HM-Churn & Amazon-LTV)](#3-relbench)
4. [Smoke Tests (Quick Sanity Checks)](#4-smoke)
5. [Ablation Studies](#5-ablations)
6. [Results Summary](#6-results)

---
## 0. Setup & Imports <a id='0-setup'></a>

Set `PROJECT_ROOT` to the **absolute path** of the `edgemamba3/` directory.  
All modules will be imported from there.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  👇 SET YOUR DIRECTORY PATHS HERE                               ║
# ╚══════════════════════════════════════════════════════════════════╝

PROJECT_ROOT = r""   # <-- Paste edgemamba3/ path here
OUTPUT_DIR   = r""   # <-- Paste output path here (e.g. Kaggle's /kaggle/working)

import os
# If left empty, auto-detect from notebook location
if not PROJECT_ROOT:
    PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
    if not os.path.isfile(os.path.join(PROJECT_ROOT, "pyproject.toml")):
        raise ValueError(
            "Could not auto-detect project root.\n"
            "Please set PROJECT_ROOT manually in the cell above."
        )

# If no output dir specified, use project root
if not OUTPUT_DIR:
    OUTPUT_DIR = PROJECT_ROOT

print(f"📂 Project root: {PROJECT_ROOT}")
print(f"💾 Output dir:   {OUTPUT_DIR}")
assert os.path.isfile(os.path.join(PROJECT_ROOT, "pyproject.toml")), \
    f"pyproject.toml not found in {PROJECT_ROOT} — is this the right directory?"

# ── Set cache dir for Kaggle read-only file systems ──────────────
os.environ["EDGEMAMBA_DATA_CACHE"] = os.path.join(OUTPUT_DIR, "data_cache")


### 0.1 Install Non-Standard Dependencies

Kaggle kernels come with common libraries (numpy, pandas, torch, sklearn, etc.) pre-installed.  
The cell below installs **everything else** EdgeMamba-3 needs. Run it once per session.

In [ ]:
# ======================================================================
# Install non-standard dependencies (run once per Kaggle session)
# ======================================================================
# IMPORTANT: Restart the kernel after running this cell!

import subprocess, sys, os

def pip_install(*args, critical=True):
    """Run pip install. If critical=False, failures are warnings not errors."""
    cmd = [sys.executable, "-m", "pip", "install", *args]
    label = " ".join(a for a in args if not a.startswith("-"))[:60]
    print(f"  {label} ... ", end="", flush=True)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print("OK")
    elif critical:
        print("FAILED")
        print(f"    {result.stderr.strip()[-500:]}")
        raise RuntimeError(f"Failed to install: {label}")
    else:
        print("SKIPPED (non-fatal)")
        if result.stderr.strip():
            print(f"    {result.stderr.strip()[-300:]}")

# ── 0. Snapshot Kaggle's numpy version before anything touches it ────
import numpy as np
ORIG_NUMPY = np.__version__
print(f"Original numpy: {ORIG_NUMPY}\n")

# ── 1. Detect PyTorch + CUDA for compatible PyG wheels ──────────────
import torch

# ── PyTorch 2.6+ Compatibility ────────────────────────────────────
# Temporarily bypass weights_only=True security restrictions for PyG datasets
import functools
import torch
import torch_geometric
if hasattr(torch.serialization, "add_safe_globals"):
    try:
        torch.serialization.add_safe_globals([torch_geometric.data.data.Data])
        torch.serialization.add_safe_globals([torch_geometric.data.InMemoryDataset])
    except Exception:
        pass
    
    # Ultimate fallback for PyTorch Geometric nested unpickling
    original_load = torch.load
    def patched_load(*args, **kwargs):
        if "weights_only" not in kwargs:
            kwargs["weights_only"] = False
        return original_load(*args, **kwargs)
    torch.load = patched_load
TORCH_VER = torch.__version__.split("+")[0]
CUDA_TAG = torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
PYG_URL = f"https://data.pyg.org/whl/torch-{TORCH_VER}+cu{CUDA_TAG}.html"
print(f"PyTorch {TORCH_VER} | CUDA: cu{CUDA_TAG}")
print(f"PyG wheel index: {PYG_URL}\n")

# ── 2. Build tools ──────────────────────────────────────────────────
print("-- Build tools --")
pip_install("-q", "ninja", "packaging")

# ── 3. PyTorch Geometric ecosystem ──────────────────────────────────
print("\n-- PyTorch Geometric ecosystem --")
pip_install("-q", "torch-scatter", "-f", PYG_URL)
pip_install("-q", "torch-sparse",  "-f", PYG_URL)
pip_install("-q", "torch-cluster", "-f", PYG_URL)
pip_install("-q", "torch-spline-conv", "-f", PYG_URL)
pip_install("-q", "pyg-lib", "-f", PYG_URL)
pip_install("-q", "torch-geometric==2.5.3")

# ── 4. RelBench + PyTorch Frame ─────────────────────────────────────
print("\n-- RelBench + PyTorch Frame --")
pip_install("-q", "--force-reinstall", "--no-deps", "pytorch-frame==0.2.3")
pip_install("-q", "--force-reinstall", "--no-deps", "relbench==1.1.0")
pip_install("-q", "pytorch-frame==0.2.3")

# ── 5. Mamba SSM (PyPI, with --no-deps to protect numpy) ────────────
# Mamba 3 requires MAMBA_FORCE_BUILD=TRUE and installing from source
print("\n-- Mamba SSM --")
os.environ["MAMBA_FORCE_BUILD"] = "TRUE"
pip_install("-q", "causal-conv1d>=1.4.0", "--no-build-isolation", "--no-deps", critical=False)
pip_install("-q", "mamba-ssm @ git+https://github.com/state-spaces/mamba.git", "--no-build-isolation", "--no-deps", critical=False)
# Upgrade triton AFTER project install (Mamba-3 requires triton.set_allocator and triton>=3.5.0)
pip_install("-q", "triton>=3.5.0", "--no-deps", critical=False)

# ── 6. Other dependencies ──────────────────────────────────────────
print("\n-- Other dependencies --")
pip_install("-q", "einops==0.7.0")
pip_install("-q", "optuna==3.5.0")
pip_install("-q", "wandb==0.16.3")
pip_install("-q", "huggingface_hub", "transformers")
pip_install("-q", "tilelang==0.1.8", critical=False)
pip_install("-q", "quack-kernels==0.3.1", critical=False)

# ── 7. Restore numpy if anything changed it ─────────────────────────
print("\n-- Restoring numpy --")
import importlib
importlib.invalidate_caches()
try:
    np_now = __import__("numpy").__version__
    if np_now != ORIG_NUMPY:
        print(f"  numpy changed {ORIG_NUMPY} -> {np_now}, restoring...")
        pip_install("-q", "--force-reinstall", f"numpy=={ORIG_NUMPY}")
    else:
        print(f"  numpy {ORIG_NUMPY} unchanged, OK")
except Exception:
    pip_install("-q", "--force-reinstall", f"numpy=={ORIG_NUMPY}")

# ── 8. Verify ──────────────────────────────────────────────────────
print("\n-- Verification --")
for pkg in ["torch_geometric", "torch_scatter", "relbench", "einops"]:
    try:
        __import__(pkg)
        print(f"  {pkg}: OK")
    except Exception as e:
        print(f"  {pkg}: ISSUE ({e})")

try:
    from torch_frame import stype
    print("  torch_frame.stype: OK")
except Exception as e:
    print(f"  torch_frame: ISSUE ({e})")

try:
    from mamba_ssm import Mamba2
    print("  mamba-2: OK")
    try:
        from mamba_ssm.modules.mamba3 import Mamba3
        print("  mamba-3: OK")
    except ImportError:
        print("  mamba-3: not available (Mamba-2 fallback will be used)")
except Exception as e:
    print(f"  mamba-ssm: ISSUE ({e})")

print("\n" + "="*50)
print("RESTART THE KERNEL after this cell, then continue.")
print("="*50)


In [ ]:
# ── Add project root to sys.path so all imports resolve ──────────────
import sys

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    print(f"✅ Added {PROJECT_ROOT} to sys.path")

# Change working directory so relative paths in configs/scripts work
os.chdir(PROJECT_ROOT)
print(f"📁 Working directory: {os.getcwd()}")

In [ ]:
# ── Core imports ────────────────────────────────────────────────────
import torch
import yaml
import time
import numpy as np
import warnings
from pathlib import Path
from datetime import datetime
from IPython.display import display, HTML, Markdown

warnings.filterwarnings("ignore", category=UserWarning)

# ── EdgeMamba-3 module imports ────────────────────────────────────────
from data.lrgb_loader import load_lrgb, compute_pos_weight
from data.relbench_loader import load_relbench
from models.edgemamba3 import EdgeMamba3
from models.line_graph import DualEmbedding, build_line_graph, compute_graph_distances, warmup_cache
from models.ltas import LTAS
from models.mamba3_encoder import BidirectionalMamba3
from models.readout import AttentionPool, TaskHead
from models.temporal_order import temporal_order
from train.trainer import Trainer
from train.distributed import run_experiment
from train.metrics import compute_metric, generate_eval_report
from train.callbacks import EarlyStopping

# Baselines (for ablations)
from baselines.node_mamba3 import NodeMamba3
from baselines.attn_ranking import EdgeMamba3_AttnRank
from baselines.static_serial import EdgeMamba3_Static
from baselines.edge_mamba2 import build_edge_mamba2

# Ablation runner
from ablations.run_ablations import run_all_ablations, run_single, ABLATIONS, BASE_CONFIG, BASE_KWARGS

print("✅ All EdgeMamba-3 modules imported successfully!")

In [ ]:
# ── Device & reproducibility ────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"🖥️  Device: {DEVICE}")
if NUM_GPUS > 0:
    for i in range(NUM_GPUS):
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"          VRAM: {torch.cuda.get_device_properties(i).total_mem / 1e9:.1f} GB")
    if NUM_GPUS > 1:
        print(f"   🚀 Multi-GPU: {NUM_GPUS} GPUs detected — DDP will be used automatically")
else:
    print("   ⚠️  No GPU detected — training will be slow.")

def set_seed(seed: int = 42):
    """Set all random seeds for reproducibility."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    print(f"🎲 Seed set to {seed}")

set_seed(42)

# ── Output directories ──────────────────────────────────────────────
NUM_SEEDS = 5
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"📦 Checkpoints → {CHECKPOINT_DIR}")
print(f"📊 Results     → {RESULTS_DIR}")


In [ ]:
# ── Helper: load YAML config ────────────────────────────────────────
def load_config(config_name: str) -> dict:
    """Load a YAML config from the configs/ directory."""
    config_path = os.path.join(PROJECT_ROOT, "configs", config_name)
    with open(config_path) as f:
        cfg = yaml.safe_load(f)
    print(f"📄 Loaded config: {config_name}")
    for k, v in cfg.items():
        print(f"   {k}: {v}")
    return cfg

# ── Helper: pretty results table ────────────────────────────────────
def display_results_table(results: list):
    """Display training results as an HTML table."""
    html = """
    <table style='border-collapse:collapse; width:100%; font-family:monospace;'>
    <tr style='background:#1a1a2e; color:#e0e0e0;'>
        <th style='padding:8px; border:1px solid #333;'>Experiment</th>
        <th style='padding:8px; border:1px solid #333;'>Metric</th>
        <th style='padding:8px; border:1px solid #333;'>Mean</th>
        <th style='padding:8px; border:1px solid #333;'>Std</th>
        <th style='padding:8px; border:1px solid #333;'>Seeds</th>
    </tr>
    """
    for r in results:
        html += f"""
        <tr style='background:#16213e; color:#e0e0e0;'>
            <td style='padding:8px; border:1px solid #333;'>{r['name']}</td>
            <td style='padding:8px; border:1px solid #333;'>{r['metric']}</td>
            <td style='padding:8px; border:1px solid #333; font-weight:bold;'>{r['mean']:.4f}</td>
            <td style='padding:8px; border:1px solid #333;'>±{r['std']:.4f}</td>
            <td style='padding:8px; border:1px solid #333;'>{r.get('seeds', 'N/A')}</td>
        </tr>
        """
    html += "</table>"
    display(HTML(html))

print("✅ Helpers loaded.")

---
## 1. Environment Validation <a id='1-env'></a>

Verify all critical and optional dependencies before starting training.

In [ ]:
# ── Environment Validation ──────────────────────────────────────────

def check_dependency(name: str, import_fn):
    """Check if a dependency is available."""
    try:
        result = import_fn()
        print(f"  ✅ {name}: {result}")
        return True
    except Exception as e:
        print(f"  ❌ {name}: FAILED — {e}")
        return False

print("=" * 60)
print("🔍 EdgeMamba-3 Environment Validation")
print("=" * 60)

print("\n── Critical Dependencies ──")
critical = [
    ("Python version",    lambda: sys.version.split()[0]),
    ("PyTorch",           lambda: torch.__version__),
    ("CUDA available",    lambda: str(torch.cuda.is_available())),
    ("torch_geometric",   lambda: __import__("torch_geometric").__version__),
    ("torch_scatter",     lambda: __import__("torch_scatter").__version__),
    ("PyYAML",            lambda: __import__("yaml").__version__),
    ("scikit-learn",      lambda: __import__("sklearn").__version__),
    ("numpy",             lambda: np.__version__),
    ("tqdm",              lambda: __import__("tqdm").__version__),
]

critical_ok = sum(check_dependency(n, f) for n, f in critical)

print("\n── Optional Dependencies ──")
optional = [
    ("Mamba-SSM",     lambda: __import__("mamba_ssm").__version__),
    ("Mamba-3",       lambda: str(__import__("mamba_ssm.modules.mamba3", fromlist=["Mamba3"]))),
    ("relbench",      lambda: __import__("relbench").__version__),
    ("torch_frame",   lambda: __import__("torch_frame").__version__),
    ("wandb",         lambda: __import__("wandb").__version__),
    ("matplotlib",    lambda: __import__("matplotlib").__version__),
    ("seaborn",       lambda: __import__("seaborn").__version__),
]

optional_ok = sum(check_dependency(n, f) for n, f in optional)

print(f"\n✅ {critical_ok}/{len(critical)} critical | {optional_ok}/{len(optional)} optional")
if critical_ok == len(critical):
    print("🟢 Environment ready for training.")
else:
    print("🔴 CRITICAL dependencies missing. Fix before proceeding.")

In [ ]:
# ── Verify project file structure ───────────────────────────────────
expected_files = [
    "models/edgemamba3.py",
    "models/line_graph.py",
    "models/ltas.py",
    "models/mamba3_encoder.py",
    "models/readout.py",
    "models/temporal_order.py",
    "data/lrgb_loader.py",
    "data/relbench_loader.py",
    "train/trainer.py",
    "train/metrics.py",
    "train/callbacks.py",
    "baselines/node_mamba3.py",
    "baselines/attn_ranking.py",
    "baselines/static_serial.py",
    "baselines/edge_mamba2.py",
    "ablations/run_ablations.py",
    "configs/lrgb_peptides_func.yaml",
    "configs/lrgb_peptides_struct.yaml",
    "configs/relbench_amazon_ltv.yaml",
    "configs/relbench_hm_churn.yaml",
]

print("📁 Verifying project files...")
missing = [f for f in expected_files if not os.path.isfile(os.path.join(PROJECT_ROOT, f))]
if missing:
    print(f"❌ Missing files:")
    for m in missing:
        print(f"   • {m}")
else:
    print(f"✅ All {len(expected_files)} expected files present.")

---
## 2. LRGB Training <a id='2-lrgb'></a>

Train EdgeMamba-3 on the **Long-Range Graph Benchmark** datasets:
- **Peptides-func**: Multi-label graph classification (AP metric)
- **Peptides-struct**: Graph-level regression (MAE metric)

Each experiment runs **5 seeds** for statistical significance.

### 2.1 Peptides-func (Classification)

In [ ]:
# ── Load config ─────────────────────────────────────────────────────
cfg_func = load_config("lrgb_peptides_func.yaml")

In [ ]:
# ── Train Peptides-func (5 seeds) ───────────────────────────────────
# Uses DDP automatically when multiple GPUs are available
lrgb_func_results = run_experiment(
    config=cfg_func,
    domain="lrgb",
    seeds=range(NUM_SEEDS),
    checkpoint_dir=CHECKPOINT_DIR,
    results_dir=RESULTS_DIR,
)

In [ ]:
# ── Peptides-func Summary ───────────────────────────────────────────
test_scores = [r["test"] for r in lrgb_func_results]
mean_ap = np.mean(test_scores)
std_ap = np.std(test_scores)

display(Markdown(f"""
### 📊 Peptides-func Results

| Metric | Value |
|--------|-------|
| **AP (Mean ± Std)** | **{mean_ap:.4f} ± {std_ap:.4f}** |
| Individual seeds | {[round(s, 4) for s in test_scores]} |
| Total training time | {sum(r['time_min'] for r in lrgb_func_results):.1f} min |
"""))

### 2.2 Peptides-struct (Regression)

In [ ]:
# ── Load config ─────────────────────────────────────────────────────
cfg_struct = load_config("lrgb_peptides_struct.yaml")

In [ ]:
# ── Train Peptides-struct (5 seeds) ─────────────────────────────────
# Uses DDP automatically when multiple GPUs are available
lrgb_struct_results = run_experiment(
    config=cfg_struct,
    domain="lrgb",
    seeds=range(NUM_SEEDS),
    checkpoint_dir=CHECKPOINT_DIR,
    results_dir=RESULTS_DIR,
)

In [ ]:
# ── Peptides-struct Summary ─────────────────────────────────────────
test_scores_struct = [r["test"] for r in lrgb_struct_results]
mean_mae = np.mean(test_scores_struct)
std_mae = np.std(test_scores_struct)

display(Markdown(f"""
### 📊 Peptides-struct Results

| Metric | Value |
|--------|-------|
| **MAE (Mean ± Std)** | **{mean_mae:.4f} ± {std_mae:.4f}** |
| Individual seeds | {[round(s, 4) for s in test_scores_struct]} |
| Total training time | {sum(r['time_min'] for r in lrgb_struct_results):.1f} min |
"""))

---
## 3. RelBench Training <a id='3-relbench'></a>

Train EdgeMamba-3 on the **Relational Benchmark** datasets:
- **rel-hm/user-churn**: Binary classification (AUROC metric)
- **rel-amazon/user-ltv**: Regression (MAE metric)

Each experiment runs **5 seeds** for statistical significance.

### 3.1 H&M User Churn (Binary Classification)

In [ ]:
# ── Load config ─────────────────────────────────────────────────────
cfg_hm = load_config("relbench_hm_churn.yaml")

In [ ]:
# ── Train H&M User Churn (5 seeds) ─────────────────────────────────
# Uses DDP automatically when multiple GPUs are available
rb_hm_results = run_experiment(
    config=cfg_hm,
    domain="relbench",
    seeds=range(NUM_SEEDS),
    checkpoint_dir=CHECKPOINT_DIR,
    results_dir=RESULTS_DIR,
)

In [ ]:
# ── H&M Churn Summary ───────────────────────────────────────────────
test_scores_hm = [r["test"] for r in rb_hm_results]
mean_auroc = np.mean(test_scores_hm)
std_auroc = np.std(test_scores_hm)

display(Markdown(f"""
### 📊 rel-hm/user-churn Results

| Metric | Value |
|--------|-------|
| **AUROC (Mean ± Std)** | **{mean_auroc:.4f} ± {std_auroc:.4f}** |
| Individual seeds | {[round(s, 4) for s in test_scores_hm]} |
| Total training time | {sum(r['time_min'] for r in rb_hm_results):.1f} min |
"""))

### 3.2 Amazon User LTV (Regression)

In [ ]:
# ── Load config ─────────────────────────────────────────────────────
cfg_amazon = load_config("relbench_amazon_ltv.yaml")

In [ ]:
# ── Train Amazon LTV (5 seeds) ─────────────────────────────────────
# Uses DDP automatically when multiple GPUs are available
rb_amazon_results = run_experiment(
    config=cfg_amazon,
    domain="relbench",
    seeds=range(NUM_SEEDS),
    checkpoint_dir=CHECKPOINT_DIR,
    results_dir=RESULTS_DIR,
)

In [ ]:
# ── Amazon LTV Summary ──────────────────────────────────────────────
test_scores_amazon = [r["test"] for r in rb_amazon_results]
mean_mae_amazon = np.mean(test_scores_amazon)
std_mae_amazon = np.std(test_scores_amazon)

display(Markdown(f"""
### 📊 rel-amazon/user-ltv Results

| Metric | Value |
|--------|-------|
| **MAE (Mean ± Std)** | **{mean_mae_amazon:.4f} ± {std_mae_amazon:.4f}** |
| Individual seeds | {[round(s, 4) for s in test_scores_amazon]} |
| Total training time | {sum(r['time_min'] for r in rb_amazon_results):.1f} min |
"""))

---
## 4. Smoke Tests (Quick Sanity Checks) <a id='4-smoke'></a>

Run fast 3-epoch smoke tests to verify end-to-end pipeline correctness  
without waiting for full convergence. Use these before long training runs.

### 4.1 LRGB Smoke Test

In [ ]:
# ── LRGB Smoke Test ─────────────────────────────────────────────────
print("🔥 LRGB Smoke Test (3 epochs, tiny model, 5 batches)")
print("=" * 50)

set_seed(42)

# Load data with small batch
smoke_train, smoke_val, _, smoke_meta = load_lrgb("Peptides-func", batch_size=4)

# Tiny model for speed
smoke_model_lrgb = EdgeMamba3(
    domain="lrgb",
    node_in_dim=smoke_meta["node_dim"],
    edge_in_dim=smoke_meta["edge_dim"],
    d_model=64,
    n_layers=1,
    d_state=16,
    mimo_rank=1,
    num_outputs=smoke_meta["num_outputs"],
    task_type=smoke_meta["task_type"],
)

smoke_config = {"lr": 1e-3, "epochs": 3, "patience": 100, "batch_size": 4, "limit_batches": 5}
smoke_trainer = Trainer(smoke_model_lrgb, smoke_config, device=DEVICE)

t0 = time.time()
smoke_best = smoke_trainer.fit(
    smoke_train, smoke_val,
    domain="lrgb", metric=smoke_meta["metric"],
    save_path=os.path.join(CHECKPOINT_DIR, "smoke_lrgb.pt"),
)

smoke_test_score = smoke_trainer.test(
    smoke_val, domain="lrgb",
    metric=smoke_meta["metric"],
    checkpoint_path=os.path.join(CHECKPOINT_DIR, "smoke_lrgb.pt"),
)

elapsed = time.time() - t0
print(f"\n{'='*50}")
if smoke_best > 0.0:
    print(f"✅ LRGB Smoke Test PASSED — AP={smoke_best:.4f}, Test={smoke_test_score:.4f} ({elapsed:.1f}s)")
else:
    print(f"❌ LRGB Smoke Test FAILED — AP={smoke_best:.4f}")

### 4.2 RelBench Smoke Test

In [ ]:
# ── RelBench Smoke Test ─────────────────────────────────────────────
print("🔥 RelBench Smoke Test (3 epochs, tiny model, 5 batches)")
print("=" * 50)

set_seed(42)

smoke_rb_train, smoke_rb_val, _, smoke_rb_meta = load_relbench(
    "rel-hm/user-churn", batch_size=16, num_workers=0
)

smoke_model_rb = EdgeMamba3(
    domain="relbench",
    event_feat_dim=smoke_rb_meta["event_feat_dim"],
    d_model=32,
    n_layers=1,
    d_state=16,
    mimo_rank=1,
    num_outputs=smoke_rb_meta["num_outputs"],
    task_type=smoke_rb_meta["task_type"],
)

smoke_rb_config = {"lr": 1e-3, "epochs": 3, "patience": 100, "batch_size": 16, "limit_batches": 5}
smoke_rb_trainer = Trainer(smoke_model_rb, smoke_rb_config, device=DEVICE)

t0 = time.time()
smoke_rb_best = smoke_rb_trainer.fit(
    smoke_rb_train, smoke_rb_val,
    domain="relbench", metric=smoke_rb_meta["metric"],
    save_path=os.path.join(CHECKPOINT_DIR, "smoke_relbench.pt"),
)

smoke_rb_test = smoke_rb_trainer.test(
    smoke_rb_val, domain="relbench",
    metric=smoke_rb_meta["metric"],
    checkpoint_path=os.path.join(CHECKPOINT_DIR, "smoke_relbench.pt"),
)

elapsed = time.time() - t0
print(f"\n{'='*50}")
if smoke_rb_best > 0.0:
    print(f"✅ RelBench Smoke Test PASSED — AUROC={smoke_rb_best:.4f}, Test={smoke_rb_test:.4f} ({elapsed:.1f}s)")
else:
    print(f"❌ RelBench Smoke Test FAILED — AUROC={smoke_rb_best:.4f}")

---
## 5. Ablation Studies <a id='5-ablations'></a>

Run the full ablation suite covering:
- **RQ1**: Edge-centric vs Node-centric (EdgeMamba-3 vs NodeMamba-3)
- **RQ2**: Serialization quality (LTAS vs Attention Ranking vs BFS vs DFS vs Random)
- **RQ3**: SSM backbone (Mamba-3 MIMO vs SISO vs Mamba-2)
- **RQ4**: Distance encoding impact

> ⚠️ **Warning**: Running all ablations takes a long time. 
> Consider setting `N_SEEDS_ABLATION = 1` for a quick preview.

In [ ]:
# ── Configuration ───────────────────────────────────────────────────
N_SEEDS_ABLATION = 5   # Set to 1 for quick test, 5 for full results

print(f"⚙️  Ablation config: {N_SEEDS_ABLATION} seed(s) per experiment")
print(f"   Total experiments: {len(ABLATIONS)}")
print(f"   Total runs: {len(ABLATIONS) * N_SEEDS_ABLATION}")

In [ ]:
# ── Run All Ablations ───────────────────────────────────────────────
ablation_output_csv = os.path.join(RESULTS_DIR, "ablation_results.csv")

print("🧪 Starting ablation studies...")
print("=" * 60)

ablation_results = run_all_ablations(
    n_seeds=N_SEEDS_ABLATION,
    output_csv=ablation_output_csv
)

print(f"\n📄 Results saved to: {ablation_output_csv}")

In [ ]:
# ── Display Ablation Results ────────────────────────────────────────
import csv

if os.path.isfile(ablation_output_csv):
    with open(ablation_output_csv) as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    
    ablation_display = []
    for row in rows:
        ablation_display.append({
            "name": row["description"],
            "metric": "AP",
            "mean": float(row["mean"]),
            "std": float(row["std"]),
            "seeds": row["seeds"],
        })
    
    display(Markdown("### 🧪 Ablation Study Results"))
    display_results_table(ablation_display)
else:
    print("⚠️  No ablation results found. Run the ablation cell above first.")

---
## 6. Results Summary <a id='6-results'></a>

Consolidated results from all experiments.

In [ ]:
# ── Consolidated Results ────────────────────────────────────────────

all_results = []

# LRGB results
if 'lrgb_func_results' in dir() and lrgb_func_results:
    func_tests = [r["test"] for r in lrgb_func_results]
    all_results.append({
        "name": "Peptides-func",
        "metric": "AP ↑",
        "mean": np.mean(func_tests),
        "std": np.std(func_tests),
        "seeds": str([round(s, 4) for s in func_tests]),
    })

if 'lrgb_struct_results' in dir() and lrgb_struct_results:
    struct_tests = [r["test"] for r in lrgb_struct_results]
    all_results.append({
        "name": "Peptides-struct",
        "metric": "MAE ↓",
        "mean": np.mean(struct_tests),
        "std": np.std(struct_tests),
        "seeds": str([round(s, 4) for s in struct_tests]),
    })

# RelBench results
if 'rb_hm_results' in dir() and rb_hm_results:
    hm_tests = [r["test"] for r in rb_hm_results]
    all_results.append({
        "name": "rel-hm/user-churn",
        "metric": "AUROC ↑",
        "mean": np.mean(hm_tests),
        "std": np.std(hm_tests),
        "seeds": str([round(s, 4) for s in hm_tests]),
    })

if 'rb_amazon_results' in dir() and rb_amazon_results:
    amazon_tests = [r["test"] for r in rb_amazon_results]
    all_results.append({
        "name": "rel-amazon/user-ltv",
        "metric": "MAE ↓",
        "mean": np.mean(amazon_tests),
        "std": np.std(amazon_tests),
        "seeds": str([round(s, 4) for s in amazon_tests]),
    })

if all_results:
    display(Markdown("## 📊 EdgeMamba-3 — Final Results"))
    display_results_table(all_results)
else:
    print("⚠️  No training results available yet. Run training cells above.")

In [ ]:
# ── Save consolidated results to file ───────────────────────────────
if all_results:
    summary_path = os.path.join(RESULTS_DIR, "final_results_summary.txt")
    with open(summary_path, "w") as f:
        f.write("EdgeMamba-3 — Final Results Summary\n")
        f.write(f"Generated: {datetime.now().isoformat()}\n")
        f.write("=" * 60 + "\n\n")
        for r in all_results:
            f.write(f"{r['name']} ({r['metric']}): {r['mean']:.4f} ± {r['std']:.4f}\n")
            f.write(f"  Seeds: {r['seeds']}\n\n")
    print(f"💾 Summary saved to: {summary_path}")

In [ ]:
# ── List all generated artifacts ────────────────────────────────────
print("\n📁 Generated Artifacts")
print("=" * 60)

for dir_name, dir_path in [("Checkpoints", CHECKPOINT_DIR), ("Results", RESULTS_DIR)]:
    print(f"\n📂 {dir_name}: {dir_path}")
    if os.path.isdir(dir_path):
        files = sorted(os.listdir(dir_path))
        for f in files:
            fp = os.path.join(dir_path, f)
            size_mb = os.path.getsize(fp) / (1024 * 1024)
            print(f"   • {f} ({size_mb:.2f} MB)")
    else:
        print("   (directory not found)")

print(f"\n✅ Pipeline complete! ({datetime.now().strftime('%Y-%m-%d %H:%M:%S')})")